In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

In [ ]:
# ============================================================
# DBSCAN OUTPUT ROOT
# ============================================================

DBSCAN_OUT_ROOT = Path("/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset")


# ============================================================
# PILIH FILE UNTUK DIVISUALISASIKAN
# ============================================================
# Pilihan dataset: "development" atau "testing"

DATASET_MODE = "development"

ACTIVITY = "Jatuh"
SUBJECT = "Teguh"
FILE_ID = 1

# Khusus testing
ROOM = "Controlled Room"


# ============================================================
# BUILD FILE PATH
# ============================================================

if DATASET_MODE == "development":
    CHECK_FILE_PATH = (
        DBSCAN_OUT_ROOT
        / "Dataset Development"
        / ACTIVITY
        / SUBJECT
        / f"{FILE_ID}.csv"
    )

elif DATASET_MODE == "testing":
    CHECK_FILE_PATH = (
        DBSCAN_OUT_ROOT
        / "Dataset Testing"
        / ROOM
        / ACTIVITY
        / SUBJECT
        / f"{FILE_ID}.csv"
    )

else:
    raise ValueError("DATASET_MODE harus 'development' atau 'testing'.")


print("===== DBSCAN VISUALIZATION CONFIG =====")
print(f"Dataset mode : {DATASET_MODE}")
print(f"File path    : {CHECK_FILE_PATH}")
print(f"File exists  : {CHECK_FILE_PATH.exists()}")

In [ ]:
REQUIRED_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

if not CHECK_FILE_PATH.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {CHECK_FILE_PATH}")

df = pd.read_csv(CHECK_FILE_PATH)

missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan: {missing_cols}")

df = df[REQUIRED_COLUMNS].copy()

for col in ["frame_id", "X_corr", "Y_corr", "Z_corr", "Z_level"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["frame_id", "X_corr", "Y_corr", "Z_level"]).copy()
df["frame_id"] = df["frame_id"].astype(int)

frame_ids = sorted(df["frame_id"].unique().tolist())

print("===== FILE LOADED =====")
print(f"Shape       : {df.shape}")
print(f"Total frame : {len(frame_ids)}")
print(f"First frame : {frame_ids[0] if frame_ids else None}")
print(f"Last frame  : {frame_ids[-1] if frame_ids else None}")

display(df.head())

In [ ]:
frame_summary_df = (
    df.groupby("frame_id")
    .agg(
        n_points=("frame_id", "count"),
        timestamp_min=("Timestamp", "min"),
        timestamp_max=("Timestamp", "max"),
        x_min=("X_corr", "min"),
        x_max=("X_corr", "max"),
        y_min=("Y_corr", "min"),
        y_max=("Y_corr", "max"),
        z_min=("Z_level", "min"),
        z_max=("Z_level", "max"),
        z_mean=("Z_level", "mean"),
    )
    .reset_index()
)

frame_summary_df["x_range"] = frame_summary_df["x_max"] - frame_summary_df["x_min"]
frame_summary_df["y_range"] = frame_summary_df["y_max"] - frame_summary_df["y_min"]
frame_summary_df["z_range"] = frame_summary_df["z_max"] - frame_summary_df["z_min"]

print("===== FILE SUMMARY =====")
print(f"Total points      : {len(df)}")
print(f"Total frames      : {len(frame_summary_df)}")
print(f"Mean points/frame : {frame_summary_df['n_points'].mean():.2f}")
print(f"Median points     : {frame_summary_df['n_points'].median():.2f}")
print(f"Min points/frame  : {frame_summary_df['n_points'].min()}")
print(f"Max points/frame  : {frame_summary_df['n_points'].max()}")

display(frame_summary_df.head())

In [ ]:
ROI_X_MIN, ROI_X_MAX = 0.0, 3.0
ROI_Y_MIN, ROI_Y_MAX = -1.5, 1.5
ROI_Z_MIN, ROI_Z_MAX = 0.0, 2.0

MAX_POINTS_PLOT = 15000


def make_frame_figure(frame_id):
    frame_df = df[df["frame_id"] == frame_id].copy()

    if len(frame_df) > MAX_POINTS_PLOT:
        frame_df = frame_df.sample(MAX_POINTS_PLOT, random_state=42)

    fig = go.Figure()

    if len(frame_df) > 0:
        fig.add_trace(
            go.Scatter3d(
                x=frame_df["X_corr"],
                y=frame_df["Y_corr"],
                z=frame_df["Z_level"],
                mode="markers",
                marker=dict(
                    size=2,
                    color=frame_df["Z_level"],
                    colorscale="Turbo",
                    opacity=0.85,
                    colorbar=dict(title="Z_level"),
                ),
                text=[
                    f"frame_id={fid}<br>"
                    f"X_corr={x:.3f}<br>"
                    f"Y_corr={y:.3f}<br>"
                    f"Z_level={z:.3f}<br>"
                    f"Reflectivity={r}"
                    for fid, x, y, z, r in zip(
                        frame_df["frame_id"],
                        frame_df["X_corr"],
                        frame_df["Y_corr"],
                        frame_df["Z_level"],
                        frame_df["Reflectivity"],
                    )
                ],
                hoverinfo="text",
                name="Main cluster",
            )
        )

    fig.update_layout(
        title=(
            f"DBSCAN Main Cluster Visualization<br>"
            f"Frame ID: {frame_id} | Points: {len(frame_df)}"
        ),
        scene=dict(
            xaxis=dict(title="X_corr", range=[ROI_X_MIN, ROI_X_MAX]),
            yaxis=dict(title="Y_corr", range=[ROI_Y_MIN, ROI_Y_MAX]),
            zaxis=dict(title="Z_level", range=[ROI_Z_MIN, ROI_Z_MAX]),
            aspectmode="manual",
            aspectratio=dict(x=1.5, y=1.5, z=1.0),
        ),
        width=950,
        height=750,
        margin=dict(l=0, r=0, b=0, t=90),
    )

    return fig

In [ ]:
def make_summary_html(frame_id):
    row = frame_summary_df[frame_summary_df["frame_id"] == frame_id].iloc[0]

    rows = [
        ("frame_id", frame_id),
        ("n_points", row["n_points"]),
        ("timestamp_min", row["timestamp_min"]),
        ("timestamp_max", row["timestamp_max"]),
        ("x_range", row["x_range"]),
        ("y_range", row["y_range"]),
        ("z_range", row["z_range"]),
        ("z_min", row["z_min"]),
        ("z_max", row["z_max"]),
        ("z_mean", row["z_mean"]),
    ]

    html = """
    <div style="font-family: Arial; font-size: 13px;">
    <h3>Frame Summary - Main Cluster Only</h3>
    <table style="border-collapse: collapse;">
    """

    for key, value in rows:
        if isinstance(value, float):
            value_str = f"{value:.6f}"
        else:
            value_str = str(value)

        html += f"""
        <tr>
            <td style="border: 1px solid #ccc; padding: 4px 8px;"><b>{key}</b></td>
            <td style="border: 1px solid #ccc; padding: 4px 8px;">{value_str}</td>
        </tr>
        """

    html += """
    </table>
    </div>
    """

    return HTML(html)

In [ ]:
if len(frame_ids) == 0:
    raise ValueError("Tidak ada frame untuk divisualisasikan.")

frame_index_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    description="Frame idx:",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)

play_widget = widgets.Play(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    interval=500,
    description="Play",
    disabled=False,
)

widgets.jslink((play_widget, "value"), (frame_index_slider, "value"))

prev_button = widgets.Button(
    description="Prev Frame",
    tooltip="Go to previous frame",
    icon="arrow-left",
)

next_button = widgets.Button(
    description="Next Frame",
    tooltip="Go to next frame",
    icon="arrow-right",
)

frame_label = widgets.HTML()

plot_output = widgets.Output()
summary_output = widgets.Output()


def update_frame_label():
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]
    frame_label.value = (
        f"<b>Current:</b> index {idx + 1}/{len(frame_ids)} | "
        f"<b>frame_id:</b> {frame_id}"
    )


def render_current_frame(change=None):
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]

    update_frame_label()

    with plot_output:
        clear_output(wait=True)
        fig = make_frame_figure(frame_id)
        fig.show()

    with summary_output:
        clear_output(wait=True)
        display(make_summary_html(frame_id))


def on_prev_clicked(button):
    current = frame_index_slider.value
    if current > frame_index_slider.min:
        frame_index_slider.value = current - 1


def on_next_clicked(button):
    current = frame_index_slider.value
    if current < frame_index_slider.max:
        frame_index_slider.value = current + 1


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

frame_index_slider.observe(render_current_frame, names="value")

controls = widgets.VBox([
    widgets.HBox([play_widget, frame_index_slider]),
    widgets.HBox([prev_button, next_button]),
    frame_label,
])

display(controls)
display(summary_output)
display(plot_output)

render_current_frame()

In [ ]:
print("===== FRAMES WITH LOWEST POINT COUNTS =====")
display(
    frame_summary_df
    .sort_values("n_points", ascending=True)
    .head(15)
)

print("===== FRAMES WITH HIGHEST POINT COUNTS =====")
display(
    frame_summary_df
    .sort_values("n_points", ascending=False)
    .head(15)
)

In [ ]:
def jump_to_frame_id(target_frame_id):
    if target_frame_id not in frame_ids:
        print(f"Frame ID {target_frame_id} tidak ditemukan.")
        print(f"Available range: {frame_ids[0]} sampai {frame_ids[-1]}")
        return

    idx = frame_ids.index(target_frame_id)
    frame_index_slider.value = idx
    print(f"Jumped to frame_id={target_frame_id}, index={idx}")


# Contoh:
# jump_to_frame_id(31)